# SAR Narrative Generator Agent – Demo

This notebook demonstrates the **Narrative Generator Agent** (CrewAI) for creating the mandatory narrative section of Suspicious Activity Reports (SAR). The agent:

- Accepts input in JSON format (case, subject, alert, SuspiciousActivityInformation, transactions).
- Uses **few-shot examples** and **reference narratives** (with effectiveness guidelines) to produce factual, non-hallucinated narratives.
- Outputs the **same structure as input** with one new field added: `narrative`. So the result is input JSON + `{"narrative": "..."}`.

**Requirements:** Set `OPENAI_API_KEY` in your environment (or in a `.env` file in the project root) before running the agent cells.

## 1. Setup: Add project to path and load input example

In [ ]:
import os

os.environ["OPENAI_API_KEY"] = "sk-proj"  # or load from .env

In [8]:
import json
import sys
from pathlib import Path

# Add project src so we can import narrative_agent
project_root = Path("..").resolve()
src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

# Load the provided input example
input_path = project_root / "examples" / "input_example.json"
with open(input_path, "r") as f:
    input_example = json.load(f)

print("Case ID:", input_example["case_id"])
print("Subject:", input_example["subject"]["name"])
print("Alert red flags:", input_example["alert"]["red_flags"])
print("Transactions count:", len(input_example["transactions"]))

Case ID: CASE-2024-677021
Subject: Global Trade Corp
Alert red flags: ['Structuring']
Transactions count: 3


## 2. Few-shot examples (used by the agent)

The agent uses these examples in its prompt to learn how to translate suspicious activity data into narrative form.

In [9]:
from narrative_agent.examples import FEW_SHOT_EXAMPLES, get_few_shot_text

print(f"Number of few-shot examples: {len(FEW_SHOT_EXAMPLES)}")
for i, ex in enumerate(FEW_SHOT_EXAMPLES, 1):
    print(f"\n--- Example {i} ---")
    print("Input keys:", list(ex["input_snippet"].keys()))
    print("Narrative (first 120 chars):", ex["narrative"][:120] + "...")

Number of few-shot examples: 3

--- Example 1 ---
Input keys: ['case_id', 'amount', 'date_from', 'date_to', 'subject', 'alert', 'categories', 'activities']
Narrative (first 120 chars): 
This Suspicious Activity Report (SAR) is filed to report suspected structuring and money laundering activities involvin...

--- Example 2 ---
Input keys: ['case_id', 'amount', 'date_from', 'date_to', 'subject', 'alert', 'categories', 'activities']
Narrative (first 120 chars): 
A review of recent account activity associated with Maria Thompson identified indicators consistent with potential acco...

--- Example 3 ---
Input keys: ['case_id', 'amount', 'date_from', 'date_to', 'subject', 'alert', 'categories', 'activities']
Narrative (first 120 chars): 
Unusual international payment activity conducted by Global Imports LLC prompted further investigation into potential tr...


## 3. Reference narratives and effectiveness guidelines

The agent references these guidelines to produce narratives suitable for SAR reports.

In [10]:
from narrative_agent.narrative_reference import (
    REFERENCE_NARRATIVES,
    EFFECTIVENESS_GUIDELINES,
    get_reference_context,
)

print("Effectiveness guidelines (excerpt):")
print(EFFECTIVENESS_GUIDELINES[:500], "...")
print("\nReference narratives:")
for ref in REFERENCE_NARRATIVES:
    print(f"  - {ref['summary']}: {ref['effectiveness'][:80]}...")

Effectiveness guidelines (excerpt):

SAR narrative effectiveness guidelines (use when generating):

1. Use only information explicitly provided in the input. Do not add names, dates, amounts, or facts not present.
2. Maintain a factual and objective tone. Do not conclude that a crime occurred; describe observed activity and patterns.
3. Include specific transactional detail: dates, amounts, transaction counts, account identifiers (if provided), and total aggregates where applicable.
4. Clearly connect suspicious patterns to the un ...

Reference narratives:
  - Structured cash deposits followed by immediate international wire transfers to a single beneficiary.: Effective because it clearly documents structured deposits with specific dates a...
  - Corporate accounts exhibiting repeated structured cash deposits, high-volume wires, and foreign correspondent banking involvement.: Effective because it systematically outlines suspicious patterns including struc...
  - Grocery store owner en

## 4. Run the Narrative Generator Agent

Run the agent on the currently loaded input. The agent returns the **same JSON as input** with one new key `narrative` added.

In [11]:
# Optionally (re)load a specific input for the single run below. Default: input_example.json
input_path = project_root / "examples" / "input_example.json"
with open(input_path, "r") as f:
    input_example = json.load(f)
print("Loaded:", input_path.name, "| Case ID:", input_example["case_id"])

Loaded: input_example.json | Case ID: CASE-2024-677021


In [6]:
from narrative_agent import generate_narrative

output = generate_narrative(input_example, verbose=True)
# Output = same as input with one new field "narrative" (all input keys preserved)
input_keys = [k for k in output.keys() if k != "narrative"]
print("\n--- Output: input + narrative (same format as input, one new field) ---")
print("Input keys preserved:", input_keys)
print("New field added: narrative")
print("\nFull output (complete JSON = input + narrative):")
print(json.dumps(output, indent=2))

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  8eb7c048-b78f-4125-a90c-96a769ffb75b                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: You are generating the mandatory narrative section for a Suspicious Activity Report (SAR). You must NOT  │
│  hallucinate or invent any information. Use ONLY the data provided below.                                       │
│                                                                                                                 │
│                                                                                                                 │
│  SAR narrative effectiveness guidelines (use when generating):                                                  │
│                                                                                                                 │
│  1. Use only information explicitly provided in the input. Do not add names, dates, amounts, or facts not       │
│  present.                                                                                                       │
│  2. Maintain a factual and objective tone. Do not conclude that a crime occurred; describe observed activity    │
│  and patterns.                                                                                                  │
│  3. Include specific transactional detail: dates, amounts, transaction counts, account identifiers (if          │
│  provided), and total aggregates where applicable.                                                              │
│  4. Clearly connect suspicious patterns to the underlying data (e.g., structuring, rapid movement of funds,     │
│  foreign wires, correspondent banking activity).                                                                │
│  5. Where applicable, document institutional actions taken (e.g., CTR filing, account closure, law enforcement  │
│  contact, ongoing investigation).                                                                               │
│  6. Identify deviations from expected customer behavior or stated business purpose when supported by the        │
│  input.                                                                                                         │
│  7. Organize logically: subject identification → time frame → transactional details → suspicious patterns →     │
│  institutional response.                                                                                        │
│  8. Avoid emotional or accusatory language. Use neutral phrasing such as "appears inconsistent," "raises        │
│  concern," or "may indicate."                                                                                   │
│  9. Output a single continuous narrative paragraph suitable for the SAR narrative section.                      │
│  10. Return the narrative in JSON format as: {"narrative": "..."}.                                              │
│                                                                                                                 │
│                                                                                                                 │
│  Reference examples (what makes narratives effective):                                                          │
│    1. Structured cash deposits followed by immediate international wire transfers to a single beneficiary.:     │
│  Effective because it clearly documents structured deposits with specific dates and amounts, links deposits to  │
│  subsequent wire transfers, and explains why the activity deviates from the customer’s stated occupation. It    │
│  includes the internal bank reference number, identifies the beneficiary and destination bank, states that      │
│  investigation is continuing, and specifies where suppo

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: SAR Narrative Writer                                                                                    │
│                                                                                                                 │
│  Task: You are generating the mandatory narrative section for a Suspicious Activity Report (SAR). You must NOT  │
│  hallucinate or invent any information. Use ONLY the data provided below.                                       │
│                                                                                                                 │
│                                                                                                                 │
│  SAR narrative effectiveness guidelines (use when generating):                                                  │
│                                                                                                                 │
│  1. Use only information explicitly provided in the input. Do not add names, dates, amounts, or facts not       │
│  present.                                                                                                       │
│  2. Maintain a factual and objective tone. Do not conclude that a crime occurred; describe observed activity    │
│  and patterns.                                                                                                  │
│  3. Include specific transactional detail: dates, amounts, transaction counts, account identifiers (if          │
│  provided), and total aggregates where applicable.                                                              │
│  4. Clearly connect suspicious patterns to the underlying data (e.g., structuring, rapid movement of funds,     │
│  foreign wires, correspondent banking activity).                                                                │
│  5. Where applicable, document institutional actions taken (e.g., CTR filing, account closure, law enforcement  │
│  contact, ongoing investigation).                                                                               │
│  6. Identify deviations from expected customer behavior or stated business purpose when supported by the        │
│  input.                                                                                                         │
│  7. Organize logically: subject identification → time frame → transactional details → suspicious patterns →     │
│  institutional response.                                                                                        │
│  8. Avoid emotional or accusatory language. Use neutral phrasing such as "appears inconsistent," "raises        │
│  concern," or "may indicate."                                                                                   │
│  9. Output a single continuous narrative paragraph suitable for the SAR narrative section.                      │
│  10. Return the narrative in JSON format as: {"narrative": "..."}.                                              │
│                                                                                                                 │
│                                                                                                                 │
│  Reference examples (what makes narratives effective):                                                          │
│    1. Structured cash deposits followed by immediate international wire transfers to a single beneficiary.:     │
│  Effective because it clearly documents structured deposits with specific dates and amounts, links deposits to  │
│  subsequent wire transfers, and explains why the activity deviates from the customer’s stated occupation. It    │
│  includes the internal bank reference number, identifie

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: SAR Narrative Writer                                                                                    │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  {"narrative": "This Suspicious Activity Report (SAR) is filed to report potential structuring and money        │
│  laundering activities involving Global Trade Corp, an individual with a Medium risk rating. The activity was   │
│  identified through Alert ALRT-78051 generated by the ML Risk Model v2 due to unusual cash patterns and         │
│  multiple day deposits. Between March 15, 2024, and March 22, 2024, the subject engaged in a series of cash     │
│  deposits and funds transfers totaling approximately $25,500. On March 15, 2024, at 10:30 AM, a cash deposit    │
│  of $9,500 was made at a location in Miami, FL, followed by an immediate wire transfer. On March 18, 2024, at   │
│  2:00 PM, another cash deposit of $8,000 was recorded, which was followed by a smaller wire transfer. Finally,  │
│  on March 22, 2024, at 9:15 AM, a funds transfer of $8,000 was executed, bringing the total activity just       │
│  below the Currency Transaction Report (CTR) threshold. The pattern of multiple cash deposits followed by       │
│  immediate wire transfers, along with the use of different destination accounts, raises concerns of             │
│  structuring and may indicate attempts to evade regulatory reporting requirements. Additionally, the subject    │
│  declined to explain the large cash deposits, and there is adverse media indicating an ongoing investigation.   │
│  This activity warrants further regulatory review."}                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  You are generating the mandatory narrative section for a Suspicious Activity Report (SAR). You must NOT        │
│  hallucinate or invent any information. Use ONLY the data provided below.                                       │
│                                                                                                                 │
│                                                                                                                 │
│  SAR narrative effectiveness guidelines (use when generating):                                                  │
│                                                                                                                 │
│  1. Use only information explicitly provided in the input. Do not add names, dates, amounts, or facts not       │
│  present.                                                                                                       │
│  2. Maintain a factual and objective tone. Do not conclude that a crime occurred; describe observed activity    │
│  and patterns.                                                                                                  │
│  3. Include specific transactional detail: dates, amounts, transaction counts, account identifiers (if          │
│  provided), and total aggregates where applicable.                                                              │
│  4. Clearly connect suspicious patterns to the underlying data (e.g., structuring, rapid movement of funds,     │
│  foreign wires, correspondent banking activity).                                                                │
│  5. Where applicable, document institutional actions taken (e.g., CTR filing, account closure, law enforcement  │
│  contact, ongoing investigation).                                                                               │
│  6. Identify deviations from expected customer behavior or stated business purpose when supported by the        │
│  input.                                                                                                         │
│  7. Organize logically: subject identification → time frame → transactional details → suspicious patterns →     │
│  institutional response.                                                                                        │
│  8. Avoid emotional or accusatory language. Use neutral phrasing such as "appears inconsistent," "raises        │
│  concern," or "may indicate."                                                                                   │
│  9. Output a single continuous narrative paragraph suitable for the SAR narrative section.                      │
│  10. Return the narrative in JSON format as: {"narrative": "..."}.                                              │
│                                                                                                                 │
│                                                                                                                 │
│  Reference examples (what makes narratives effective):                                                          │
│    1. Structured cash deposits followed by immediate international wire transfers to a single beneficiary.:     │
│  Effective because it clearly documents structured deposits with specific dates and amounts, links deposits to  │
│  subsequent wire transfers, and explains why the activity deviates from the customer’s stated occupation. It    │
│  includes the internal bank reference number, identifie

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  8eb7c048-b78f-4125-a90c-96a769ffb75b                                                                           │
│  Final Output: {"narrative": "This Suspicious Activity Report (SAR) is filed to report potential structuring    │
│  and money laundering activities involving Global Trade Corp, an individual with a Medium risk rating. The      │
│  activity was identified through Alert ALRT-78051 generated by the ML Risk Model v2 due to unusual cash         │
│  patterns and multiple day deposits. Between March 15, 2024, and March 22, 2024, the subject engaged in a       │
│  series of cash deposits and funds transfers totaling approximately $25,500. On March 15, 2024, at 10:30 AM, a  │
│  cash deposit of $9,500 was made at a location in Miami, FL, followed by an immediate wire transfer. On March   │
│  18, 2024, at 2:00 PM, another cash deposit of $8,000 was recorded, which was followed by a smaller wire        │
│  transfer. Finally, on March 22, 2024, at 9:15 AM, a funds transfer of $8,000 was executed, bringing the total  │
│  activity just below the Currency Transaction Report (CTR) threshold. The pattern of multiple cash deposits     │
│  followed by immediate wire transfers, along with the use of different destination accounts, raises concerns    │
│  of structuring and may indicate attempts to evade regulatory reporting requirements. Additionally, the         │
│  subject declined to explain the large cash deposits, and there is adverse media indicating an ongoing          │
│  investigation. This activity warrants further regulatory review."}                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


--- Output: input + narrative (same format as input, one new field) ---
Input keys preserved: ['case_id', 'report_type', 'generated_at', 'subject', 'institution', 'alert', 'external_signals', 'data_quality', 'SuspiciousActivityInformation', 'transactions']
New field added: narrative

Full output (complete JSON = input + narrative):
{
  "case_id": "CASE-2024-677021",
  "report_type": "Initial",
  "generated_at": "2025-11-18T03:42:03Z",
  "subject": {
    "subject_id": "C-94926",
    "name": "Global Trade Corp",
    "type": "Individual",
    "risk_rating": "Medium",
    "country": "US",
    "onboarding_date": "2022-05-09",
    "industry_or_occupation": "Retail",
    "beneficial_owners": [
      "Owner A (60%)",
      "Owner B (40%)"
    ],
    "prior_sars": [
      "2022-372022"
    ]
  },
  "institution": {
    "name": "Example Community Bank",
    "branch_city": "New York",
    "branch_state": "NY",
    "contact_officer": "Brian Lee",
    "contact_phone": "+1-800-XXX-XXXX",
    "prima

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [7]:
import textwrap
from narrative_agent import generate_narrative

INPUT_EXAMPLE_NAMES = [
    "input_example.json",
    # "input_example2.json",
    # "input_example3.json",
]
examples_dir = project_root / "examples"
width = 80

for filename in INPUT_EXAMPLE_NAMES:
    path = examples_dir / filename
    if not path.exists():
        print(f"Skipping {filename} (not found)")
        continue
    with open(path, "r") as f:
        data = json.load(f)
    print("=" * 60)
    print(f"Input file: {filename}")
    print(f"Case ID: {data.get('case_id', 'N/A')}")
    print("=" * 60)
    output = generate_narrative(data, verbose=False)
    # Output has same keys as input + "narrative"
    assert "narrative" in output, "Missing 'narrative' in output"
    print("Generated narrative (wrapped):")
    print("-" * 40)
    print(textwrap.fill(output["narrative"], width=width))
    print("-" * 40)
    print()

Input file: input_example.json
Case ID: CASE-2024-677021
Generated narrative (wrapped):
----------------------------------------
This Suspicious Activity Report (SAR) is filed to report suspicious cash deposit
and funds transfer activities involving Global Trade Corp, an individual with a
Medium risk rating. The activity was identified through Alert ALRT-78051
generated by the ML Risk Model v2 due to unusual cash patterns and multiple day
deposits. Between March 15, 2024, and March 22, 2024, the subject engaged in a
series of cash deposits and wire transfers totaling approximately $25,500. On
March 15, 2024, at 10:30 AM, a cash deposit of $9,500 was made from account
XXXXXX0001 in Miami, FL, followed by an immediate wire transfer. On March 18,
2024, at 2:00 PM, another cash deposit of $8,000 was recorded from the same
account, which was followed by a smaller wire transfer. Finally, on March 22,
2024, at 9:15 AM, a funds transfer of $8,000 was executed from account
XXXXXX0001 to another

In [8]:
# Full output = input + narrative (same format as input, one new field). Print so nothing is truncated:
print("Full output (input + narrative):")
print(json.dumps(output, indent=2))

Full output (input + narrative):
{
  "case_id": "CASE-2024-677021",
  "report_type": "Initial",
  "generated_at": "2025-11-18T03:42:03Z",
  "subject": {
    "subject_id": "C-94926",
    "name": "Global Trade Corp",
    "type": "Individual",
    "risk_rating": "Medium",
    "country": "US",
    "onboarding_date": "2022-05-09",
    "industry_or_occupation": "Retail",
    "beneficial_owners": [
      "Owner A (60%)",
      "Owner B (40%)"
    ],
    "prior_sars": [
      "2022-372022"
    ]
  },
  "institution": {
    "name": "Example Community Bank",
    "branch_city": "New York",
    "branch_state": "NY",
    "contact_officer": "Brian Lee",
    "contact_phone": "+1-800-XXX-XXXX",
    "primary_federal_regulator": "FDIC"
  },
  "alert": {
    "alert_id": "ALRT-78051",
    "alert_date": "2025-01-11",
    "model_name": "ML Risk Model v2",
    "trigger_reasons": [
      "Unusual cash pattern",
      "Multiple day deposits"
    ],
    "risk_score": 0.41,
    "red_flags": [
      "Structuring"

## 5. Display the generated narrative

The narrative is the mandatory section content for the SAR report. It is available as `output["narrative"]`; the rest of `output` is the original input.

In [9]:
import textwrap

narrative_text = output["narrative"]
width = 80
print("Generated SAR narrative:")
print("-" * 60)
print(textwrap.fill(narrative_text, width=width))
print("-" * 60)

Generated SAR narrative:
------------------------------------------------------------
This Suspicious Activity Report (SAR) is filed to report suspicious cash deposit
and funds transfer activities involving Global Trade Corp, an individual with a
Medium risk rating. The activity was identified through Alert ALRT-78051
generated by the ML Risk Model v2 due to unusual cash patterns and multiple day
deposits. Between March 15, 2024, and March 22, 2024, the subject engaged in a
series of cash deposits and wire transfers totaling approximately $25,500. On
March 15, 2024, at 10:30 AM, a cash deposit of $9,500 was made from account
XXXXXX0001 in Miami, FL, followed by an immediate wire transfer. On March 18,
2024, at 2:00 PM, another cash deposit of $8,000 was recorded from the same
account, which was followed by a smaller wire transfer. Finally, on March 22,
2024, at 9:15 AM, a funds transfer of $8,000 was executed from account
XXXXXX0001 to another account, bringing the total activity just 

## 6. Test with all 3 input examples

Run the agent on each of the three example JSON files by name and display the generated narrative for each.

In [14]:
import textwrap
from narrative_agent import generate_narrative

INPUT_EXAMPLE_NAMES = [
    "input_example.json",
    "input_example2.json",
    "input_example3.json",
]
examples_dir = project_root / "examples"
width = 80

for filename in INPUT_EXAMPLE_NAMES:
    path = examples_dir / filename
    if not path.exists():
        print(f"Skipping {filename} (not found)")
        continue
    with open(path, "r") as f:
        data = json.load(f)
    print("=" * 60)
    print(f"Input file: {filename}")
    print(f"Case ID: {data.get('case_id', 'N/A')}")
    print("=" * 60)
    output = generate_narrative(data, verbose=False)
    # Output has same keys as input + "narrative"
    assert "narrative" in output, "Missing 'narrative' in output"
    print(output)
    # print("Generated narrative (wrapped):")
    # print("-" * 40)
    # print(textwrap.fill(output["narrative"], width=width))
    # print("-" * 40)
    # print()

Input file: input_example.json
Case ID: CASE-2024-677021
{'case_id': 'CASE-2024-677021', 'report_type': 'Initial', 'generated_at': '2025-11-18T03:42:03Z', 'subject': {'subject_id': 'C-94926', 'name': 'Global Trade Corp', 'type': 'Individual', 'risk_rating': 'Medium', 'country': 'US', 'onboarding_date': '2022-05-09', 'industry_or_occupation': 'Retail', 'beneficial_owners': ['Owner A (60%)', 'Owner B (40%)'], 'prior_sars': ['2022-372022']}, 'institution': {'name': 'Example Community Bank', 'branch_city': 'New York', 'branch_state': 'NY', 'contact_officer': 'Brian Lee', 'contact_phone': '+1-800-XXX-XXXX', 'primary_federal_regulator': 'FDIC'}, 'alert': {'alert_id': 'ALRT-78051', 'alert_date': '2025-01-11', 'model_name': 'ML Risk Model v2', 'trigger_reasons': ['Unusual cash pattern', 'Multiple day deposits'], 'risk_score': 0.41, 'red_flags': ['Structuring']}, 'external_signals': {'adverse_media': ['News report: subject under investigation'], 'law_enforcement_contacted': {'contacted': False,

In [13]:
output

{'case_id': 'CASE-2025-425659',
 'report_type': 'Initial',
 'generated_at': '2025-11-18T03:41:19Z',
 'subject': {'subject_id': 'C-72530',
  'name': 'Global Trade Corp',
  'type': 'Business',
  'risk_rating': 'Low',
  'country': 'US',
  'onboarding_date': '2017-10-25',
  'industry_or_occupation': 'Import/Export',
  'beneficial_owners': ['Owner A (60%)', 'Owner B (40%)'],
  'prior_sars': ['2022-636221']},
 'institution': {'name': 'Example Community Bank',
  'branch_city': 'Charlotte',
  'branch_state': 'NY',
  'contact_officer': 'Brian Lee',
  'contact_phone': '+1-800-XXX-XXXX',
  'primary_federal_regulator': 'NCUA'},
 'alert': {'alert_id': 'ALRT-68434',
  'alert_date': '2024-04-13',
  'model_name': 'ML Risk Model v2',
  'trigger_reasons': ['Unusual cash pattern', 'Multiple day deposits'],
  'risk_score': 0.7,
  'red_flags': ['Structuring']},
 'external_signals': {'adverse_media': ['News report: subject under investigation'],
  'law_enforcement_contacted': {'contacted': False,
   'agency